# Summit 2026 HOL — Snowflake Postgres + Transactional Data
## Module 09 (Optional): Add a Transactional Layer with Market Data

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🛠️ Creates a **Snowflake Postgres instance** (shows the feature)
2. 🛠️ **Immediately suspends** it (cost protection)
3. 🛠️ Loads **market index data** into the TRANSACTIONS schema (simulates CDC output)
4. 🛠️ Loads **live order flow** data (simulates OLTP order queue)
5. 🛠️ **Extends the semantic view** to include market data
6. 🛠️ Tests the agent with **market-aware questions**

---

### The Production Story:

```
┌──────────────────────┐     CDC / Openflow      ┌──────────────────────┐
│  Snowflake Postgres  │ ──────────────────────► │  Snowflake Tables    │
│  (OLTP: Orders,      │   Near real-time        │  (TRANSACTIONS       │
│   Market Feeds)      │   replication           │   schema)            │
└──────────────────────┘                         └─────────┬────────────┘
                                                           │
                                                           ▼
                                                 ┌──────────────────────┐
                                                 │  Semantic View +     │
                                                 │  Cortex Agent        │
                                                 │  (queries both       │
                                                 │   analytics AND      │
                                                 │   transactional)     │
                                                 └──────────────────────┘
```

In production, your OLTP application writes to Snowflake Postgres. The **Openflow Connector for PostgreSQL** replicates changes into Snowflake tables via CDC in near real-time. For this lab, we create the Postgres instance to demonstrate the feature, then simulate the CDC output by loading data directly.

---

### Estimated Time: **15–20 minutes**

### Prerequisites:
- Modules 01–03 complete (NEXUS_HOL database, semantic view, agent)
- ACCOUNTADMIN access (for Postgres instance creation)

## Step 1: Set Context

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE DATABASE NEXUS_HOL;
USE WAREHOUSE NEXUS_WH;

📌 Switch to **ACCOUNTADMIN** in the Snowsight role selector (bottom-left corner) — the Postgres UI uses your active Snowsight role, not the notebook session role.

## Step 2: 🛠️ Create the Snowflake Postgres Instance (via UI)

### This demonstrates Snowflake Postgres — a fully managed PostgreSQL database running inside Snowflake.

🔹 **Use case:** Your trading application writes live orders and market data to Postgres. Snowflake's Openflow Connector replicates this data into Snowflake tables via CDC, where your Cortex Agent can query it alongside analytics data.

---

### 🛠️ Create the instance using Snowsight:

1. In the left navigation, click the **Postgres** icon (elephant head 🐘)
2. Click **+ Create** (top right)
3. Configure:
   - **Name:** `NEXUS_POSTGRES`
   - **Compute Family:** `STANDARD_M` (1 core, 4GB — smallest available)
   - **Storage:** `10 GB`
   - **Postgres Version:** `18`
   - **Network Policy:** Leave blank (we won't connect to it)
4. Click **Create**
5. ⚠️ **Save the credentials** displayed — they won't be shown again (though we won't use them in this lab)

📌 **Instance creation takes 2–4 minutes.** The status will progress through: CREATING → RESTORING → STARTING → FINALIZING → READY. Wait until you see **READY** before proceeding to the next step. You can refresh the page or run the SHOW POSTGRES INSTANCES cell below to check.

📌 **Cost note:** We'll suspend immediately in the next step. While running, even idle, this instance bills continuously (~1 credit/hour for STANDARD_M).

In [ ]:
%%sql -r ShowPostgres
-- Verify the Postgres instance exists (run after creating via UI)
SHOW POSTGRES INSTANCES;

In [ ]:
%%sql -r dataframe_3
-- Check instance status (run once UI shows READY)
-- If you get 'does not exist', wait for the UI creation to complete and re-run
DESCRIBE POSTGRES INSTANCE NEXUS_POSTGRES
  ->> SELECT "property", "value"
      FROM $1
      WHERE "property" IN ('name', 'state', 'host', 'compute_family', 'storage_size_gb', 'postgres_version');

## Step 3: 🛠️ Suspend Immediately (Cost Protection)

### ⚠️ Snowflake Postgres bills continuously — unlike warehouses, there is NO auto-suspend.

📌 **Key difference from warehouses:**

| | Warehouse | Postgres Instance |
|---|---|---|
| Billing | Only when running queries | **Continuous** while running |
| Auto-suspend | ✅ Yes (configurable) | ❌ No — must suspend manually |
| Cost when idle | $0 | Credits/hour + storage |
| Resume time | ~1 second | 3-5 minutes |

We suspend now to protect against forgotten cleanup. The instance retains its data on disk — you can resume it anytime.

📌 **Alternative:** You can also suspend via the UI: Postgres → select instance → Manage → Suspend.

In [ ]:
%%sql -r dataframe_4
-- Suspend immediately to stop billing
ALTER POSTGRES INSTANCE IF EXISTS NEXUS_POSTGRES SUSPEND;

SELECT 'Postgres instance NEXUS_POSTGRES suspended (no ongoing compute charges)' AS STATUS;

## Step 4: The Production Architecture

🔹 In a real deployment, the flow would be:

1. **Trading app** writes orders and market feeds to the Postgres instance
2. **Openflow Connector for PostgreSQL** captures changes via CDC (logical replication)
3. **Snowflake tables** receive near real-time replicated data in the TRANSACTIONS schema
4. **Cortex Agent** queries both ANALYTICS (historical) and TRANSACTIONS (live) data

📌 **For this lab**, we skip the CDC step and load sample data directly into the TRANSACTIONS schema — this is exactly what the replicated tables would look like after the connector runs.

> **Documentation:** [Openflow Connector for PostgreSQL](https://docs.snowflake.com/en/user-guide/data-integration/openflow/connectors/postgres/about) | [Snowflake Postgres](https://docs.snowflake.com/en/user-guide/snowflake-postgres/about)

---

Now let's load the data that represents what CDC would replicate.

## Step 5: 🛠️ Load Market Index Data

### Create and populate the MARKET_INDICES table

This represents daily closing prices for major indices and sector ETFs that a trading desk monitors. In production, a market data feed writes this to Postgres; here we generate 30 days of realistic data.

| Symbol | Index/ETF | Category |
|--------|-----------|----------|
| SPY | S&P 500 ETF | Broad Market |
| QQQ | NASDAQ-100 ETF | Broad Market |
| DIA | Dow Jones ETF | Broad Market |
| IWM | Russell 2000 ETF | Broad Market |
| XLK | Technology Select Sector | Sector ETF |
| XLF | Financial Select Sector | Sector ETF |
| XLE | Energy Select Sector | Sector ETF |
| XLV | Health Care Select Sector | Sector ETF |

In [ ]:
%%sql -r dataframe_5
USE ROLE NEXUS_ADMIN;
USE SCHEMA NEXUS_HOL.TRANSACTIONS;

In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE TABLE NEXUS_HOL.TRANSACTIONS.MARKET_INDICES (
    INDEX_DATE          DATE NOT NULL,
    INDEX_SYMBOL        VARCHAR(10) NOT NULL,
    INDEX_NAME          VARCHAR(50) NOT NULL,
    CLOSE_PRICE         NUMBER(12,2) NOT NULL,
    DAILY_CHANGE_PCT    NUMBER(6,3),
    VOLUME              NUMBER(15) ,
    YTD_RETURN_PCT      NUMBER(6,2),
    CATEGORY            VARCHAR(20) NOT NULL,
    PRIMARY KEY (INDEX_DATE, INDEX_SYMBOL)
);

SELECT 'MARKET_INDICES table created' AS STATUS;

In [ ]:
%%sql -r dataframe_7
-- Generate 30 days of market data for 8 indices/ETFs
-- Uses realistic base prices with small random daily moves
INSERT INTO NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
WITH date_series AS (
    SELECT DATEADD('day', -seq4(), CURRENT_DATE()) AS INDEX_DATE
    FROM TABLE(GENERATOR(ROWCOUNT => 30))
),
indices AS (
    SELECT * FROM VALUES
        ('SPY', 'S&P 500 ETF', 545.00, 'Broad Market'),
        ('QQQ', 'NASDAQ-100 ETF', 482.00, 'Broad Market'),
        ('DIA', 'Dow Jones ETF', 425.00, 'Broad Market'),
        ('IWM', 'Russell 2000 ETF', 228.00, 'Broad Market'),
        ('XLK', 'Technology Select Sector', 215.00, 'Sector ETF'),
        ('XLF', 'Financial Select Sector', 44.50, 'Sector ETF'),
        ('XLE', 'Energy Select Sector', 89.00, 'Sector ETF'),
        ('XLV', 'Health Care Select Sector', 148.00, 'Sector ETF')
    AS t(SYMBOL, NAME, BASE_PRICE, CATEGORY)
)
SELECT
    d.INDEX_DATE,
    i.SYMBOL,
    i.NAME,
    ROUND(i.BASE_PRICE * (1 + (ABS(HASH(i.SYMBOL || d.INDEX_DATE::VARCHAR)) % 600 - 300) / 10000.0), 2) AS CLOSE_PRICE,
    ROUND((ABS(HASH(d.INDEX_DATE::VARCHAR || i.SYMBOL)) % 400 - 200) / 100.0, 3) AS DAILY_CHANGE_PCT,
    ABS(HASH(i.SYMBOL || d.INDEX_DATE::VARCHAR || 'vol')) % 80000000 + 20000000 AS VOLUME,
    ROUND((ABS(HASH(i.SYMBOL || 'ytd')) % 2400 - 400) / 100.0, 2) AS YTD_RETURN_PCT,
    i.CATEGORY
FROM date_series d
CROSS JOIN indices i
ORDER BY d.INDEX_DATE DESC, i.SYMBOL;

SELECT 'Market indices loaded: ' || COUNT(*) || ' rows (30 days x 8 indices)' AS STATUS
FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES;

In [ ]:
%%sql -r dataframe_8
-- Preview: Latest day market data
SELECT INDEX_SYMBOL, INDEX_NAME, CLOSE_PRICE, DAILY_CHANGE_PCT, YTD_RETURN_PCT, CATEGORY
FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
WHERE INDEX_DATE = CURRENT_DATE()
ORDER BY CATEGORY, INDEX_SYMBOL;

## Step 6: 🛠️ Load Live Order Flow

### Create the ORDER_FLOW table — simulates the OLTP order queue

In production, traders submit orders through a Postgres-backed application. Orders have lifecycle states:
- **PENDING** — Submitted, awaiting execution
- **PARTIAL_FILL** — Partially executed
- **FILLED** — Fully executed
- **CANCELLED** — Withdrawn by trader

This gives the agent visibility into *in-flight* orders — not just settled trades.

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE TABLE NEXUS_HOL.TRANSACTIONS.ORDER_FLOW (
    ORDER_ID            NUMBER AUTOINCREMENT PRIMARY KEY,
    CLIENT_ID           NUMBER NOT NULL,
    SYMBOL              VARCHAR(10) NOT NULL,
    ORDER_TYPE          VARCHAR(10) NOT NULL,       -- MARKET_BUY, MARKET_SELL, LIMIT_BUY, LIMIT_SELL
    QUANTITY            NUMBER NOT NULL,
    LIMIT_PRICE         NUMBER(12,4),               -- NULL for market orders
    ORDER_STATUS        VARCHAR(15) NOT NULL,       -- PENDING, PARTIAL_FILL, FILLED, CANCELLED
    SUBMITTED_AT        TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    FILLED_AT           TIMESTAMP_NTZ,
    FILLED_QUANTITY     NUMBER DEFAULT 0,
    NOTES               VARCHAR(200)
);

SELECT 'ORDER_FLOW table created' AS STATUS;

In [ ]:
%%sql -r dataframe_10
-- Load simulated live order flow
INSERT INTO NEXUS_HOL.TRANSACTIONS.ORDER_FLOW
    (CLIENT_ID, SYMBOL, ORDER_TYPE, QUANTITY, LIMIT_PRICE, ORDER_STATUS, SUBMITTED_AT, FILLED_AT, FILLED_QUANTITY, NOTES)
VALUES
-- Pending orders (not yet filled)
(1, 'GOOGL', 'LIMIT_BUY', 3000, 175.50, 'PENDING', DATEADD('minute', -45, CURRENT_TIMESTAMP()), NULL, 0, 'Waiting for dip below 176'),
(3, 'NVDA', 'LIMIT_SELL', 5000, 140.00, 'PENDING', DATEADD('minute', -30, CURRENT_TIMESTAMP()), NULL, 0, 'Take profit if hits 140'),
(5, 'ASML', 'LIMIT_BUY', 1500, 880.00, 'PENDING', DATEADD('minute', -20, CURRENT_TIMESTAMP()), NULL, 0, 'Adding on pullback'),
(9, 'MSTR', 'LIMIT_SELL', 2000, 1750.00, 'PENDING', DATEADD('minute', -15, CURRENT_TIMESTAMP()), NULL, 0, 'Scaling out at resistance'),

-- Partially filled
(2, 'PG', 'MARKET_BUY', 8000, NULL, 'PARTIAL_FILL', DATEADD('minute', -60, CURRENT_TIMESTAMP()), NULL, 5200, 'Large block - filling in tranches'),
(4, 'SONY', 'LIMIT_BUY', 20000, 94.00, 'PARTIAL_FILL', DATEADD('hour', -2, CURRENT_TIMESTAMP()), NULL, 12000, 'Accumulating at support'),

-- Filled today
(1, 'MSFT', 'MARKET_BUY', 4000, NULL, 'FILLED', DATEADD('hour', -3, CURRENT_TIMESTAMP()), DATEADD('hour', -3, CURRENT_TIMESTAMP()), 4000, 'Pre-earnings position'),
(3, 'AMD', 'MARKET_BUY', 8000, NULL, 'FILLED', DATEADD('hour', -2, CURRENT_TIMESTAMP()), DATEADD('hour', -2, CURRENT_TIMESTAMP()), 8000, 'AI infrastructure thesis'),
(6, 'AAPL', 'MARKET_BUY', 2500, NULL, 'FILLED', DATEADD('hour', -4, CURRENT_TIMESTAMP()), DATEADD('hour', -4, CURRENT_TIMESTAMP()), 2500, 'Adding to core'),
(8, 'BABA', 'MARKET_SELL', 15000, NULL, 'FILLED', DATEADD('hour', -1, CURRENT_TIMESTAMP()), DATEADD('hour', -1, CURRENT_TIMESTAMP()), 15000, 'Reducing China exposure'),
(10, 'SAP', 'MARKET_BUY', 3000, NULL, 'FILLED', DATEADD('hour', -5, CURRENT_TIMESTAMP()), DATEADD('hour', -5, CURRENT_TIMESTAMP()), 3000, 'Enterprise software conviction'),

-- Cancelled
(7, 'VTI', 'LIMIT_BUY', 50000, 260.00, 'CANCELLED', DATEADD('hour', -6, CURRENT_TIMESTAMP()), NULL, 0, 'Market moved away - cancelled'),
(12, '9988.HK', 'LIMIT_SELL', 30000, 115.00, 'CANCELLED', DATEADD('hour', -4, CURRENT_TIMESTAMP()), NULL, 0, 'Client changed mind on exit');

SELECT 'Order flow loaded: ' || COUNT(*) || ' orders (' ||
    SUM(CASE WHEN ORDER_STATUS = 'PENDING' THEN 1 ELSE 0 END) || ' pending, ' ||
    SUM(CASE WHEN ORDER_STATUS = 'PARTIAL_FILL' THEN 1 ELSE 0 END) || ' partial, ' ||
    SUM(CASE WHEN ORDER_STATUS = 'FILLED' THEN 1 ELSE 0 END) || ' filled, ' ||
    SUM(CASE WHEN ORDER_STATUS = 'CANCELLED' THEN 1 ELSE 0 END) || ' cancelled)' AS STATUS
FROM NEXUS_HOL.TRANSACTIONS.ORDER_FLOW;

## Step 7: 🛠️ Extend the Semantic View

### Add the MARKET_INDICES table to the existing semantic view

📌 Semantic views don't support `ALTER ... ADD TABLE` — we must re-create with all existing definitions plus the new table. This is the standard pattern for extending a semantic view.

The agent will now be able to answer questions like:
- "How is the S&P 500 performing this month?"
- "Which sector ETF is outperforming?"
- "Compare our tech exposure to XLK performance"

In [ ]:
%%sql -r dataframe_11
USE SCHEMA NEXUS_HOL.SEMANTIC;

CREATE OR REPLACE SEMANTIC VIEW NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV

  TABLES (
    clients AS NEXUS_HOL.ANALYTICS.CLIENTS
      PRIMARY KEY (CLIENT_ID)
      COMMENT = 'Client master data including institutional and HNW accounts with AUM and risk profiles',

    positions AS NEXUS_HOL.ANALYTICS.POSITIONS
      PRIMARY KEY (POSITION_ID)
      COMMENT = 'Current portfolio positions by client and symbol with market values and unrealized PnL',

    trades AS NEXUS_HOL.ANALYTICS.TRADES
      PRIMARY KEY (TRADE_ID)
      COMMENT = 'Trade execution history including buy/sell orders with prices, quantities, and status',

    market_indices AS NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
      PRIMARY KEY (INDEX_DATE, INDEX_SYMBOL)
      COMMENT = 'Daily closing prices for major market indices and sector ETFs. Replicated from Postgres via CDC.'
  )

  RELATIONSHIPS (
    positions_to_clients AS
      positions(CLIENT_ID) REFERENCES clients,
    trades_to_clients AS
      trades(CLIENT_ID) REFERENCES clients
  )

  FACTS (
    clients.AUM AS AUM
      COMMENT = 'Assets Under Management in USD. Total value of assets the client has with Nexus Capital.',
    positions.QUANTITY AS POSITION_QUANTITY
      COMMENT = 'Number of shares held in the position',
    positions.AVG_COST AS AVG_COST
      COMMENT = 'Average cost basis per share for the position',
    positions.CURRENT_PRICE AS CURRENT_PRICE
      COMMENT = 'Current market price per share',
    positions.MARKET_VALUE AS MARKET_VALUE
      COMMENT = 'Current market value of the position (quantity * current_price)',
    positions.UNREALIZED_PNL AS UNREALIZED_PNL
      COMMENT = 'Unrealized profit or loss on the position (market_value - cost_basis). Positive = gain, negative = loss.',
    trades.QUANTITY AS TRADE_QUANTITY
      COMMENT = 'Number of shares in the trade order',
    trades.PRICE AS TRADE_PRICE
      COMMENT = 'Execution price per share for the trade',
    market_indices.CLOSE_PRICE AS INDEX_CLOSE_PRICE
      COMMENT = 'Closing price of the index or ETF on the given date',
    market_indices.DAILY_CHANGE_PCT AS INDEX_DAILY_CHANGE
      COMMENT = 'Daily percentage change in the index price. Positive = up day, negative = down day.',
    market_indices.YTD_RETURN_PCT AS INDEX_YTD_RETURN
      COMMENT = 'Year-to-date return percentage for the index or ETF'
  )

  DIMENSIONS (
    clients.CLIENT_NAME AS CLIENT_NAME
      COMMENT = 'Full name of the client account',
    clients.ACCOUNT_TYPE AS ACCOUNT_TYPE
      COMMENT = 'Account classification: INSTITUTIONAL, HNW (High Net Worth), or RETAIL',
    clients.REGION AS REGION
      COMMENT = 'Geographic region: North America, EMEA, or APJ',
    clients.RISK_PROFILE AS RISK_PROFILE
      COMMENT = 'Investment risk tolerance: CONSERVATIVE, MODERATE, or AGGRESSIVE',
    clients.RELATIONSHIP_MANAGER AS RELATIONSHIP_MANAGER
      COMMENT = 'Name of the assigned relationship manager',
    clients.STATUS AS STATUS
      COMMENT = 'Client account status: ACTIVE or INACTIVE',
    clients.ONBOARDED_DATE AS ONBOARDED_DATE
      COMMENT = 'Date the client was onboarded',

    positions.SYMBOL AS SYMBOL
      COMMENT = 'Stock ticker symbol (e.g., AAPL, NVDA, MSFT)',
    positions.SECTOR AS SECTOR
      COMMENT = 'Market sector classification (Technology, Financials, Energy, Healthcare, etc.)',

    trades.TRADE_TYPE AS TRADE_TYPE
      COMMENT = 'Direction of the trade: BUY or SELL',
    trades.STATUS AS TRADE_STATUS
      COMMENT = 'Trade execution status: EXECUTED or SETTLED',
    trades.EXCHANGE AS EXCHANGE
      COMMENT = 'Exchange where trade was executed (NYSE, NASDAQ, OTC)',
    trades.TRADE_DATE AS TRADE_DATE
      COMMENT = 'Timestamp when the trade was executed',
    trades.NOTES AS TRADE_NOTES
      COMMENT = 'Trader notes explaining the rationale for the trade',

    market_indices.INDEX_SYMBOL AS INDEX_SYMBOL
      COMMENT = 'Ticker symbol for the index or ETF (SPY, QQQ, DIA, IWM, XLK, XLF, XLE, XLV)',
    market_indices.INDEX_NAME AS INDEX_NAME
      COMMENT = 'Full name of the index or ETF (e.g., S&P 500 ETF, Technology Select Sector)',
    market_indices.CATEGORY AS INDEX_CATEGORY
      COMMENT = 'Classification: Broad Market or Sector ETF',
    market_indices.INDEX_DATE AS INDEX_DATE
      COMMENT = 'Date of the market data observation'
  )

  METRICS (
    clients.TOTAL_AUM AS SUM(clients.AUM)
      COMMENT = 'Total Assets Under Management across all clients in USD',

    clients.CLIENT_COUNT AS COUNT(DISTINCT CLIENT_ID)
      COMMENT = 'Number of distinct client accounts',

    positions.TOTAL_PORTFOLIO_VALUE AS SUM(positions.MARKET_VALUE)
      COMMENT = 'Total current market value across all positions in USD',

    positions.TOTAL_UNREALIZED_PNL AS SUM(positions.UNREALIZED_PNL)
      COMMENT = 'Total unrealized profit/loss across all positions. Positive = overall portfolio gains.',

    positions.POSITION_COUNT AS COUNT(DISTINCT POSITION_ID)
      COMMENT = 'Number of distinct portfolio positions',

    positions.AVG_POSITION_VALUE AS AVG(positions.MARKET_VALUE)
      COMMENT = 'Average market value per position',

    trades.TRADE_COUNT AS COUNT(DISTINCT TRADE_ID)
      COMMENT = 'Number of trade executions',

    trades.TOTAL_TRADE_VOLUME AS SUM(trades.QUANTITY * trades.PRICE)
      COMMENT = 'Total dollar volume of trades (quantity * price summed across all trades)',

    market_indices.AVG_DAILY_CHANGE AS AVG(market_indices.DAILY_CHANGE_PCT)
      COMMENT = 'Average daily percentage change for the index over the selected period'
  )

  COMMENT = 'Nexus Capital - Financial analytics semantic view covering clients, positions, trades, and market indices'

  AI_SQL_GENERATION '
    -- Business rules for SQL generation:
    -- 1. When asked about "top clients" or "biggest clients", rank by AUM unless otherwise specified.
    -- 2. When asked about portfolio performance, use UNREALIZED_PNL. Positive = gains.
    -- 3. Default time filter: if no date specified, include all available data.
    -- 4. "Active clients" means STATUS = ''ACTIVE''.
    -- 5. When asked about "recent trades", order by TRADE_DATE DESC and limit to last 7 days unless specified.
    -- 6. Trade volume = QUANTITY * PRICE. Always compute as dollar volume, not share count.
    -- 7. For region breakdowns, use the CLIENTS.REGION column (North America, EMEA, APJ).
    -- 8. When computing concentration, use MARKET_VALUE / total portfolio MARKET_VALUE.
    -- 9. For market/index questions, use the MARKET_INDICES table. Use most recent INDEX_DATE for "current" prices.
    -- 10. To compare sector performance to benchmarks, join POSITIONS.SECTOR with the corresponding sector ETF (XLK=Technology, XLF=Financials, XLE=Energy, XLV=Healthcare).
    -- 11. "Market conditions" or "market performance" refers to the broad market indices (SPY, QQQ, DIA, IWM).
  '

  AI_VERIFIED_QUERIES (
    top_clients_by_aum AS (
      QUESTION 'What are our top 5 clients by AUM?'
      SQL 'SELECT CLIENT_NAME, ACCOUNT_TYPE, REGION, AUM, RISK_PROFILE
      FROM NEXUS_HOL.ANALYTICS.CLIENTS
      WHERE STATUS = ''ACTIVE''
      ORDER BY AUM DESC
      LIMIT 5'
    ),

    portfolio_value_by_sector AS (
      QUESTION 'What is the total portfolio value by sector?'
      SQL 'SELECT p.SECTOR, SUM(p.MARKET_VALUE) AS TOTAL_VALUE, SUM(p.UNREALIZED_PNL) AS TOTAL_PNL
      FROM NEXUS_HOL.ANALYTICS.POSITIONS p
      GROUP BY p.SECTOR
      ORDER BY TOTAL_VALUE DESC'
    ),

    recent_large_buy_trades AS (
      QUESTION 'Show me recent buy trades over $1M in value'
      SQL 'SELECT c.CLIENT_NAME, t.SYMBOL, t.QUANTITY, t.PRICE, (t.QUANTITY * t.PRICE) AS TRADE_VALUE, t.TRADE_DATE, t.NOTES
      FROM NEXUS_HOL.ANALYTICS.TRADES t
      JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON t.CLIENT_ID = c.CLIENT_ID
      WHERE t.TRADE_TYPE = ''BUY'' AND (t.QUANTITY * t.PRICE) > 1000000
      ORDER BY t.TRADE_DATE DESC'
    ),

    clients_highest_unrealized_gains AS (
      QUESTION 'Which clients have the highest unrealized gains?'
      SQL 'SELECT c.CLIENT_NAME, c.ACCOUNT_TYPE, SUM(p.UNREALIZED_PNL) AS TOTAL_UNREALIZED_PNL, SUM(p.MARKET_VALUE) AS TOTAL_PORTFOLIO_VALUE
      FROM NEXUS_HOL.ANALYTICS.CLIENTS c
      JOIN NEXUS_HOL.ANALYTICS.POSITIONS p ON c.CLIENT_ID = p.CLIENT_ID
      GROUP BY c.CLIENT_NAME, c.ACCOUNT_TYPE
      ORDER BY TOTAL_UNREALIZED_PNL DESC'
    ),

    aum_breakdown_by_region AS (
      QUESTION 'What is our AUM breakdown by region?'
      SQL 'SELECT REGION, COUNT(*) AS CLIENT_COUNT, SUM(AUM) AS TOTAL_AUM, AVG(AUM) AS AVG_AUM
      FROM NEXUS_HOL.ANALYTICS.CLIENTS
      WHERE STATUS = ''ACTIVE''
      GROUP BY REGION
      ORDER BY TOTAL_AUM DESC'
    ),

    tech_sector_exposure AS (
      QUESTION 'Show me the technology sector exposure across all clients'
      SQL 'SELECT c.CLIENT_NAME, p.SYMBOL, p.QUANTITY, p.MARKET_VALUE, p.UNREALIZED_PNL
      FROM NEXUS_HOL.ANALYTICS.POSITIONS p
      JOIN NEXUS_HOL.ANALYTICS.CLIENTS c ON p.CLIENT_ID = c.CLIENT_ID
      WHERE p.SECTOR = ''Technology''
      ORDER BY p.MARKET_VALUE DESC'
    ),

    market_performance_today AS (
      QUESTION 'How are the major market indices performing today?'
      SQL 'SELECT INDEX_SYMBOL, INDEX_NAME, CLOSE_PRICE, DAILY_CHANGE_PCT, YTD_RETURN_PCT, CATEGORY
      FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
      WHERE INDEX_DATE = (SELECT MAX(INDEX_DATE) FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES)
      ORDER BY CATEGORY, INDEX_SYMBOL'
    ),

    sector_etf_comparison AS (
      QUESTION 'Which sector ETF has the best YTD return?'
      SQL 'SELECT INDEX_SYMBOL, INDEX_NAME, YTD_RETURN_PCT, CLOSE_PRICE
      FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
      WHERE CATEGORY = ''Sector ETF'' AND INDEX_DATE = (SELECT MAX(INDEX_DATE) FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES)
      ORDER BY YTD_RETURN_PCT DESC'
    )
  );

SELECT 'Semantic view NEXUS_CAPITAL_SV extended with MARKET_INDICES' AS STATUS;

In [ ]:
%%sql -r dataframe_12
-- Confirm the semantic view now has 4 tables
SHOW SEMANTIC VIEWS IN SCHEMA NEXUS_HOL.SEMANTIC;

## Step 8: 🛠️ Test the Agent with Market Questions

### Verify the agent can now answer questions about market indices

You have two options:

---

### Option A: CoWork (Recommended — easier to read)

1. Navigate to **AI & ML → Cortex AI → CoWork**
2. Open your **Nexus Capital Analyst** SI instance
3. Try these questions:

> "How are the major market indices performing? Which sector ETF has the best YTD return?"

> "Compare our technology sector portfolio exposure to the XLK sector ETF performance. Are we outperforming the benchmark?"

You'll get a formatted response with tables and reasoning steps — much easier to interpret than raw JSON.

---

### Option B: DATA_AGENT_RUN (programmatic — raw JSON output)

Run the SQL cells below if you prefer to test directly from the notebook. The output is raw JSON — look for the `"text"` field in
the response to find the agent's answer.

In [ ]:
-- -- Test: Market performance question
-- SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
--   'NEXUS_HOL.AGENTS.NEXUS_AGENT',
--   '{"messages": [{"role": "user", "content": [{"type": "text", "text": "How are the major market indices performing? Which sector ETF has the best YTD return?"}]}], "stream": false}'
-- ) AS resp;

In [ ]:
-- -- Test: Cross-domain question (portfolio + market)
-- SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
--   'NEXUS_HOL.AGENTS.NEXUS_AGENT',
--   '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Compare our technology sector portfolio exposure to the XLK sector ETF performance. Are we outperforming or underperforming the benchmark?"}]}], "stream": false}'
-- ) AS resp;

### 📌 What Just Happened:

The agent answered market questions by querying the `MARKET_INDICES` table — data that in production would flow from Postgres via CDC. This demonstrates the full pattern:

1. **OLTP writes** happen in Postgres (orders, market feeds)
2. **CDC replicates** to Snowflake tables in near real-time
3. **Semantic view** gives the agent governed access to both analytics AND transactional data
4. **Agent answers** questions that span both layers

No external client needed. No separate ETL pipeline. One platform.

## Step 9: Validate

### Confirm all objects are in place

In [ ]:
%%sql -r dataframe_15
-- Validate transactional layer
SELECT 'MARKET_INDICES' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
UNION ALL
SELECT 'ORDER_FLOW', COUNT(*) FROM NEXUS_HOL.TRANSACTIONS.ORDER_FLOW
ORDER BY TABLE_NAME;

In [ ]:
%%sql -r dataframe_16
-- Verify Postgres instance exists (should be suspended)
USE ROLE ACCOUNTADMIN;
SHOW POSTGRES INSTANCES;

## ✅ Module 09 Complete!

### You now have:
- Snowflake Postgres instance: `NEXUS_POSTGRES` (suspended — no ongoing cost)
- Market data: `NEXUS_HOL.TRANSACTIONS.MARKET_INDICES` (30 days × 8 indices = 240 rows)
- Order flow: `NEXUS_HOL.TRANSACTIONS.ORDER_FLOW` (13 orders in various states)
- Extended semantic view with market data dimensions and metrics
- Agent can now answer market + portfolio cross-domain questions

---

### What You Just Built:

```
┌──────────────────┐       ┌──────────────────┐       ┌──────────────────┐
│ OLTP Layer       │  CDC  │ Transaction Layer│  SV   │ AI Layer         │
│                  │──────▶│                  │──────▶│                  │
│ Postgres Instance│       │ MARKET_INDICES   │       │ Semantic View    │
│ (suspended)      │       │ ORDER_FLOW       │       │ (4 tables now)   │
│                  │       │                  │       │ Cortex Agent     │
└──────────────────┘       └──────────────────┘       └──────────────────┘
```

---

### 🔄 Going Deeper: Dynamic Tables

In production, you wouldn’t query raw transactional tables directly — you’d add a **Dynamic Table** that continuously aggregates and refreshes on a schedule. Dynamic Tables eliminate the need for custom ETL pipelines or task scheduling.

```sql
-- Example: Continuous market summary, refreshed every 5 minutes
CREATE DYNAMIC TABLE NEXUS_HOL.ANALYTICS.MARKET_SUMMARY
  TARGET_LAG = '5 minutes'
  WAREHOUSE = NEXUS_WH
AS
  SELECT
    INDEX_DATE,
    CATEGORY,
    AVG(DAILY_CHANGE_PCT) AS AVG_DAILY_CHANGE,
    AVG(YTD_RETURN_PCT) AS AVG_YTD_RETURN,
    COUNT(*) AS INDEX_COUNT
  FROM NEXUS_HOL.TRANSACTIONS.MARKET_INDICES
  GROUP BY INDEX_DATE, CATEGORY;
```

This would give the agent a pre-aggregated "market conditions" table that stays fresh automatically — no cron jobs, no Airflow DAGs, no manual refresh.

> **Documentation:** [Dynamic Tables](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview)

---

### Key Talking Points:

- "Snowflake Postgres lets you run OLTP workloads on the same platform as your analytics — no separate infrastructure."
- "Openflow Connector replicates Postgres changes to Snowflake via CDC. The agent queries the replicated data without touching the OLTP system."
- "One semantic view spans both historical analytics and live transactional data — the agent doesn’t care where the data originated."
- "Postgres instances can be suspended when not needed — you only pay when they’re running."
- "Dynamic Tables add continuous transformation without custom ETL — set a target lag and Snowflake handles the rest."

---

### Cleanup:

```sql
-- Drop the Postgres instance (stops all billing)
DROP POSTGRES INSTANCE IF EXISTS NEXUS_POSTGRES;

-- Drop transactional tables (optional — no cost when idle)
DROP TABLE IF EXISTS NEXUS_HOL.TRANSACTIONS.MARKET_INDICES;
DROP TABLE IF EXISTS NEXUS_HOL.TRANSACTIONS.ORDER_FLOW;
```

⚠️ **Important:** If you skip cleanup, the Postgres instance is already suspended — it will only incur minimal storage charges. But best practice is to `DROP` it when you’re done.

---

### ✅ You’ve completed all optional modules!

Return to **Module 08** to run the final DORA validation if you haven’t already.

In [ ]:
%%sql -r DropPostgresInstance
-- Drop the Postgres instance (stops all billing)
DROP POSTGRES INSTANCE IF EXISTS NEXUS_POSTGRES;

-- Drop transactional tables (optional — no cost when idle)
DROP TABLE IF EXISTS NEXUS_HOL.TRANSACTIONS.MARKET_INDICES;
DROP TABLE IF EXISTS NEXUS_HOL.TRANSACTIONS.ORDER_FLOW;